# 🚀 Optimizador Masivo de Estrategias · VectorBT + CUDA (RTX Pro 6000 · 96 GB VRAM)

Este notebook busca, entre **cientos de miles de combinaciones de parámetros**, las estrategias de trading que ganan dinero en el pasado (*In-Sample*) **y siguen ganando** en datos que nunca vieron (*Out-Of-Sample*).

**Cómo usarlo (sin saber programar):**
1. Sube tu CSV de Dukascopy a `Google Drive → MyDrive → activos`.
2. Menú **Entorno de ejecución → Cambiar tipo de entorno** → elige la GPU.
3. Edita **solo la Celda 2** (nombre del archivo, dirección, etc.).
4. Menú **Entorno de ejecución → Ejecutar todas**.

| Celda | Qué hace |
|---|---|
| 1 | Instala librerías, monta Drive y muestra el hardware (RAM, GPU, VRAM) |
| 2 | ✏️ Tus parámetros |
| 3 | Carga el CSV en `float32`, divide IS/OOS y aplica la **purga** anti-fugas |
| 4 | Simula toda la malla en la GPU (un hilo CUDA por estrategia) |
| 5 | Descarta perdedoras, elige el Top y lo valida en OOS con VectorBT |
| 6 | Guarda el CSV de estrategias robustas en Drive |

> ⚠️ Los resultados pasados no garantizan resultados futuros. Usa esto como herramienta de investigación, no como consejo financiero.

## 1️⃣ Instalación del entorno, detección de hardware y montaje de Google Drive

Instala `vectorbt`, `cupy-cuda12x`, `numba` (+ `numba-cuda` para compilar kernels en GPUs Blackwell como la RTX Pro 6000), `pandas` y `numpy`. Después monta tu Drive y muestra un informe del hardware.

- Si aparece **"Modo GPU"** al final, todo está listo.
- Si aparece **"Modo CPU"**, el notebook funcionará igual, pero mucho más lento: revisa que el entorno de ejecución tenga la GPU seleccionada.

In [ ]:
# ==============================================================================
# CELDA 1 · INSTALACIÓN DEL ENTORNO, DETECCIÓN DE HARDWARE Y GOOGLE DRIVE
# ==============================================================================
# Esta celda:
#   1) Instala las librerías necesarias (solo la primera vez, en silencio).
#   2) Monta tu Google Drive en /content/drive.
#   3) Muestra un informe del hardware: RAM, CPU y GPU NVIDIA (con su VRAM).
#   4) Comprueba que la GPU puede compilar kernels CUDA (CuPy + Numba CUDA).
#   5) Crea (si no existe) la carpeta /content/drive/MyDrive/activos y lista los CSV.
# ==============================================================================
import os
import sys
import subprocess
import warnings

warnings.filterwarnings("ignore")

# ¿Estamos dentro de Google Colab? (si no, el notebook funciona igual en local)
EN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ


def instalar_paquetes():
    """Instala cada paquete por separado para que un fallo no bloquee al resto."""
    paquetes = [
        "vectorbt",          # Motor de backtesting vectorizado
        "cupy-cuda12x",      # NumPy sobre la GPU (CUDA 12.x → RTX Pro 6000 / Blackwell)
        "numba",             # Compilador JIT (CPU)
        "numba-cuda[cu12]",  # Compilador JIT para kernels CUDA (soporte para GPUs modernas)
        "pandas",
        "numpy",
        "psutil",            # Lectura de RAM / CPU
        "matplotlib",        # Gráficos (mapa de calor)
    ]
    print("📦 Instalando librerías (puede tardar 1-2 minutos la primera vez)...")
    for paquete in paquetes:
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", paquete],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
            )
            print(f"   ✅ {paquete}")
        except subprocess.CalledProcessError as error:
            detalle = (error.stderr or "").strip()[-300:]
            print(f"   ⚠️ No se pudo instalar '{paquete}'. Detalle: {detalle}")


if EN_COLAB:
    instalar_paquetes()
else:
    print("ℹ️ No estás en Google Colab: se asume que las librerías ya están instaladas.")

# ------------------------------------------------------------------------------
# 2) Montaje de Google Drive
# ------------------------------------------------------------------------------
if EN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUTA_ACTIVOS = "/content/drive/MyDrive/activos"
    except Exception as error:
        raise RuntimeError(
            "❌ No se pudo montar Google Drive. Vuelve a ejecutar la celda y acepta los "
            f"permisos de acceso en la ventana emergente. Detalle técnico: {error}"
        )
else:
    RUTA_ACTIVOS = os.path.abspath("activos")

# ------------------------------------------------------------------------------
# 3) Informe de hardware
# ------------------------------------------------------------------------------
import psutil

GiB = 1024 ** 3
ram = psutil.virtual_memory()
print("\n" + "=" * 70)
print("🖥️  INFORME DE HARDWARE")
print("=" * 70)
print(f"RAM del sistema     : {ram.total / GiB:,.1f} GB totales | {ram.available / GiB:,.1f} GB libres")
print(f"Núcleos de CPU      : {psutil.cpu_count(logical=True)} lógicos")

GPU_DISPONIBLE = False   # ¿Hay una GPU NVIDIA usable con CuPy?
NUMBA_CUDA_OK = False    # ¿Numba puede compilar y lanzar kernels en esa GPU?
VRAM_TOTAL_GB = 0.0

try:
    import cupy as cp
    n_gpus = cp.cuda.runtime.getDeviceCount()
    if n_gpus == 0:
        raise RuntimeError("CUDA no detecta ninguna GPU.")
    props = cp.cuda.runtime.getDeviceProperties(0)
    nombre_gpu = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    vram_libre, vram_total = cp.cuda.runtime.memGetInfo()
    VRAM_TOTAL_GB = vram_total / GiB
    # Prueba real: operación en la GPU (si falla, la GPU no es usable)
    _ = float(cp.arange(1_000_000, dtype=cp.float32).sum())
    GPU_DISPONIBLE = True
    print(f"GPU detectada       : {nombre_gpu} (x{n_gpus})")
    print(f"VRAM                : {vram_total / GiB:,.1f} GB totales | {vram_libre / GiB:,.1f} GB libres")
    print(f"Compute capability  : {props['major']}.{props['minor']}")
    print(f"Multiprocesadores   : {props['multiProcessorCount']}")
except Exception as error:
    print(f"⚠️ GPU no disponible con CuPy ({error}).")
    print("   En Colab: menú 'Entorno de ejecución' → 'Cambiar tipo de entorno' → elige la GPU.")
    print("   El notebook seguirá funcionando en CPU (mucho más lento).")

# 4) Prueba de compilación de un kernel CUDA con Numba (detecta incompatibilidades)
if GPU_DISPONIBLE:
    try:
        from numba import cuda

        @cuda.jit
        def _kernel_prueba(x):
            i = cuda.grid(1)
            if i < x.size:
                x[i] = x[i] * 2.0

        _x = cp.ones(1024, dtype=cp.float32)
        _kernel_prueba[4, 256](_x)
        cuda.synchronize()
        if float(_x.sum()) != 2048.0:
            raise RuntimeError("resultado incorrecto en el kernel de prueba")
        NUMBA_CUDA_OK = True
        print("Numba CUDA          : ✅ kernels compilados y ejecutados en la GPU")
        del _x
    except Exception as error:
        print(f"Numba CUDA          : ⚠️ no disponible ({str(error)[:200]})")
        print("   Se usará Numba en CPU paralelo para la simulación (CuPy seguirá en la GPU).")

if os.path.exists("/usr/bin/nvidia-smi") or os.system("which nvidia-smi > /dev/null 2>&1") == 0:
    print("\n--- nvidia-smi ---")
    os.system("nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv")

# ------------------------------------------------------------------------------
# 5) Carpeta de activos y listado de CSV
# ------------------------------------------------------------------------------
try:
    os.makedirs(RUTA_ACTIVOS, exist_ok=True)
    archivos_csv = sorted(f for f in os.listdir(RUTA_ACTIVOS) if f.lower().endswith(".csv"))
    print("\n" + "=" * 70)
    print(f"📁 Carpeta de activos: {RUTA_ACTIVOS}")
    if archivos_csv:
        for f in archivos_csv:
            tam_mb = os.path.getsize(os.path.join(RUTA_ACTIVOS, f)) / 1024 ** 2
            print(f"   • {f}  ({tam_mb:,.1f} MB)")
    else:
        print("   (vacía) → Sube tus CSV de Dukascopy a esta carpeta de Google Drive.")
except Exception as error:
    raise RuntimeError(f"❌ No se pudo acceder a la carpeta de activos '{RUTA_ACTIVOS}': {error}")

print("\n✅ Entorno listo." + ("  🚀 Modo GPU." if GPU_DISPONIBLE else "  🐢 Modo CPU."))

## 2️⃣ Configuración de parámetros (✏️ la única celda que debes editar)

- **`NOMBRE_ARCHIVO`**: nombre exacto del CSV dentro de `MyDrive/activos`.
- **`DIRECCION_OPERACION`**: `"COMPRAS_SOLO"` o `"VENTAS_SOLO"`.
- **`PORCENTAJE_IS`**: parte de los datos usada para optimizar (el resto se reserva para validar).
- **`COMISION_PCT`**: coste por operación (comisión + slippage).
- **`GRAN_MALLE_PARAMETROS`**: rangos de las medias móviles y del RSI. Cada combinación se prueba en paralelo en la GPU; con 96 GB de VRAM puedes subir a **millones de combinaciones** (la memoria que se usa por combinación es de solo ~64 bytes).

> 💡 Sobre `vbt.Param`: es una función de **VectorBT PRO** (de pago). Con la versión gratuita de VectorBT la malla se construye con un producto cartesiano de NumPy, que es exactamente lo mismo.

In [ ]:
# ==============================================================================
# CELDA 2 · PARÁMETROS DEL USUARIO  (✏️ SOLO MODIFICA ESTA CELDA)
# ==============================================================================
import numpy as np

# 1) Archivo CSV de Dukascopy dentro de /content/drive/MyDrive/activos
NOMBRE_ARCHIVO = "EURUSD_H1.csv"

# 2) Dirección operativa: "COMPRAS_SOLO" (solo largos) o "VENTAS_SOLO" (solo cortos)
DIRECCION_OPERACION = "COMPRAS_SOLO"

# 3) Fracción de datos para In-Sample (optimización). El resto (30%) es Out-Of-Sample.
PORCENTAJE_IS = 0.70

# 4) Comisión + slippage por operación (0.00005 = 0.005% del valor negociado)
COMISION_PCT = 0.00005

# Capital inicial ficticio (solo afecta a la escala, no a los porcentajes)
CAPITAL_INICIAL = 10_000.0

# 5) GRAN MALLA DE PARÁMETROS
#    Estrategia: cruce de Media Móvil Simple rápida sobre lenta + filtro RSI.
#      • COMPRAS: entra cuando la MA rápida cruza HACIA ARRIBA la lenta y RSI < umbral
#                 (no compra en sobrecompra). Sale con el cruce hacia abajo.
#      • VENTAS : entra cuando la MA rápida cruza HACIA ABAJO la lenta y RSI > 100 - umbral
#                 (no vende en sobreventa). Sale con el cruce hacia arriba.
#      • Un umbral de 100 equivale a "sin filtro RSI".
#    Con 96 GB de VRAM puedes ampliar estos rangos sin miedo: cada combinación es un
#    hilo de la GPU. Por defecto: ~130.000 combinaciones.
GRAN_MALLE_PARAMETROS = {
    "ma_rapida":   np.arange(2, 101, 2),      # 50 periodos: 2, 4, ..., 100
    "ma_lenta":    np.arange(20, 401, 5),     # 77 periodos: 20, 25, ..., 400
    "rsi_periodo": [7, 14, 21, 28],           # 4 periodos de RSI
    "rsi_umbral":  [55, 60, 65, 70, 75, 80, 85, 90, 100],  # 9 umbrales
}

# Filtros de descarte In-Sample (eliminan estrategias perdedoras o mediocres)
FILTROS_DESCARTE = {
    "retorno_min":       0.0,    # Retorno total debe ser > 0
    "profit_factor_min": 1.2,    # Profit Factor mínimo
    "trades_min":        30,     # Mínimo de operaciones cerradas (significancia estadística)
    "trades_max":        5_000,  # Máximo de operaciones (evita sobre-operar)
    "drawdown_max":      0.25,   # Máximo drawdown permitido (0.25 = 25%)
}

# Métrica para ordenar a las supervivientes: "sharpe", "retorno_total", "profit_factor" o "calmar"
METRICA_RANKING = "sharpe"
TOP_N = 50   # Cuántas de las mejores In-Sample pasan a la validación Out-Of-Sample

# Criterios que debe cumplir una estrategia en Out-Of-Sample para considerarse robusta
FILTROS_OOS = {
    "retorno_min":       0.0,    # Debe seguir ganando dinero en datos nunca vistos
    "profit_factor_min": 1.0,    # Y ganar más de lo que pierde
    "trades_min":        5,      # Con un mínimo de operaciones
}

# Opciones avanzadas (normalmente no hace falta tocarlas)
ELIMINAR_VELAS_SIN_VOLUMEN = True   # Quita las velas "planas" de fin de semana de Dukascopy
FRACCION_MAX_VRAM = 0.80            # Nunca usar más del 80% de la VRAM (no saturar la VM)
LOTE_MAX_COMBINACIONES = 2_000_000  # Combinaciones por lanzamiento de kernel
MOSTRAR_GRAFICOS = True             # Mapa de calor + curva de capital

# ------------------------------------------------------------------------------
# Validación de los parámetros (mensajes de error amigables)
# ------------------------------------------------------------------------------
try:
    if DIRECCION_OPERACION not in ("COMPRAS_SOLO", "VENTAS_SOLO"):
        raise ValueError('DIRECCION_OPERACION debe ser exactamente "COMPRAS_SOLO" o "VENTAS_SOLO".')
    if not 0.1 <= PORCENTAJE_IS <= 0.95:
        raise ValueError("PORCENTAJE_IS debe estar entre 0.10 y 0.95 (por ejemplo 0.70).")
    if not 0 <= COMISION_PCT < 0.05:
        raise ValueError("COMISION_PCT debe ser una fracción pequeña (ej. 0.00005).")
    if METRICA_RANKING not in ("sharpe", "retorno_total", "profit_factor", "calmar"):
        raise ValueError('METRICA_RANKING debe ser "sharpe", "retorno_total", "profit_factor" o "calmar".')
    for clave in ("ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"):
        valores = np.asarray(GRAN_MALLE_PARAMETROS[clave])
        if valores.size == 0:
            raise ValueError(f"La lista '{clave}' de GRAN_MALLE_PARAMETROS está vacía.")
        if clave != "rsi_umbral" and (valores < 2).any():
            raise ValueError(f"Los periodos de '{clave}' deben ser >= 2.")
    _umbrales = np.asarray(GRAN_MALLE_PARAMETROS["rsi_umbral"])
    if ((_umbrales <= 50) | (_umbrales > 100)).any():
        raise ValueError("Los umbrales de RSI deben estar en el rango (50, 100].")
except KeyError as error:
    raise KeyError(f"❌ Falta la clave {error} en GRAN_MALLE_PARAMETROS.")

_r = np.asarray(GRAN_MALLE_PARAMETROS["ma_rapida"])
_l = np.asarray(GRAN_MALLE_PARAMETROS["ma_lenta"])
_pares_validos = int((_r[:, None] < _l[None, :]).sum())   # solo rápida < lenta
TOTAL_COMBINACIONES = (_pares_validos * len(GRAN_MALLE_PARAMETROS["rsi_periodo"])
                       * len(GRAN_MALLE_PARAMETROS["rsi_umbral"]))

print("✅ Parámetros cargados")
print(f"   Archivo      : {NOMBRE_ARCHIVO}")
print(f"   Dirección    : {DIRECCION_OPERACION}")
print(f"   IS / OOS     : {PORCENTAJE_IS:.0%} / {1 - PORCENTAJE_IS:.0%}")
print(f"   Comisión     : {COMISION_PCT:.5%} por operación")
print(f"   Combinaciones: {TOTAL_COMBINACIONES:,} (pares MA válidos rápida<lenta: {_pares_validos:,})")

## 3️⃣ Carga optimizada en memoria y prevención de fugas de datos (*anti-data leakage*)

1. Lee el CSV de Dukascopy (formato `Gmt time` o `Local time`) y convierte los precios a **`float32`** (la mitad de memoria que `float64`).
2. Ordena cronológicamente (del más antiguo al más reciente) y elimina duplicados y velas sin volumen.
3. Divide en **`df_is`** (entrenamiento) y **`df_oos`** (validación).
4. **Purga:** los indicadores del OOS se calculan solo con datos OOS. Sus primeras velas (tantas como el periodo del indicador más largo) no tienen indicadores fiables, así que **se eliminan** del OOS y se usan solo como calentamiento. Así ningún dato del IS contamina el OOS ni al revés.

```
|────────── In-Sample (70%) ──────────|░ purga ░|──── Out-Of-Sample ────|
```

In [ ]:
# ==============================================================================
# CELDA 3 · CARGA OPTIMIZADA EN MEMORIA + DIVISIÓN IS/OOS + PURGA ANTI-FUGAS
# ==============================================================================
import csv
import gc
import os

import numpy as np
import pandas as pd


def detectar_separador(ruta):
    """Detecta si el CSV usa ',', ';' o tabulador como separador."""
    with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
        muestra = f.read(4096)
    try:
        return csv.Sniffer().sniff(muestra, delimiters=",;\t").delimiter
    except csv.Error:
        return ","


def parsear_fechas(serie):
    """Convierte la columna de fecha de Dukascopy (ej. '01.01.2020 00:00:00.000 GMT+0100')."""
    if pd.api.types.is_numeric_dtype(serie):              # Marca de tiempo en milisegundos
        return pd.to_datetime(serie, unit="ms", errors="coerce")
    texto = serie.astype(str).str.strip()
    zona = texto.str.extract(r"GMT([+-]\d{2})(\d{2})\s*$")  # Exportaciones en "Local time"
    texto = texto.str.replace(r"\s*GMT[+-]\d{4}\s*$", "", regex=True)
    fechas = pd.to_datetime(texto, format="%d.%m.%Y %H:%M:%S.%f", errors="coerce")
    if fechas.isna().mean() > 0.5:                         # Otros formatos habituales
        fechas = pd.to_datetime(texto, dayfirst=True, errors="coerce")
    if zona[0].notna().any():                              # Convertir a UTC si hay desfase
        horas = zona[0].astype(float)
        minutos = zona[1].astype(float) * np.sign(horas)
        fechas = fechas - pd.to_timedelta(horas.fillna(0), unit="h") - pd.to_timedelta(minutos.fillna(0), unit="m")
    return fechas


def cargar_csv_dukascopy(ruta):
    if not os.path.isfile(ruta):
        disponibles = [f for f in os.listdir(os.path.dirname(ruta)) if f.lower().endswith(".csv")]
        raise FileNotFoundError(
            f"❌ No se encontró '{os.path.basename(ruta)}' en {os.path.dirname(ruta)}.\n"
            f"   Archivos CSV disponibles: {disponibles or 'ninguno'}.\n"
            "   Revisa NOMBRE_ARCHIVO en la Celda 2 (mayúsculas/minúsculas incluidas)."
        )
    df = pd.read_csv(ruta, sep=detectar_separador(ruta))
    df.columns = [str(c).strip().lower() for c in df.columns]

    # Columna de fecha/hora (Dukascopy: "Gmt time" o "Local time")
    candidatas = ["gmt time", "local time", "time (utc)", "datetime", "timestamp", "date", "time", "fecha"]
    if "date" in df.columns and "time" in df.columns:
        df["datetime"] = df["date"].astype(str) + " " + df["time"].astype(str)
        candidatas = ["datetime"]
    col_fecha = next((c for c in candidatas if c in df.columns), None)
    if col_fecha is None:
        raise ValueError(f"❌ No se encontró columna de fecha. Columnas del archivo: {list(df.columns)}")

    renombres = {"o": "open", "h": "high", "l": "low", "c": "close", "v": "volume", "vol": "volume"}
    df = df.rename(columns=renombres)
    faltan = [c for c in ("open", "high", "low", "close") if c not in df.columns]
    if faltan:
        raise ValueError(f"❌ Faltan las columnas de precios {faltan}. Columnas: {list(df.columns)}")

    df.index = parsear_fechas(df[col_fecha])
    df.index.name = "fecha"
    columnas = ["open", "high", "low", "close"] + (["volume"] if "volume" in df.columns else [])
    df = df[columnas].apply(pd.to_numeric, errors="coerce").astype("float32")  # float32 = mitad de RAM

    filas_iniciales = len(df)
    df = df[df.index.notna()].dropna(subset=["open", "high", "low", "close"])
    df = df[(df[["open", "high", "low", "close"]] > 0).all(axis=1)]
    df = df.sort_index()                                   # Cronológico: antiguo → reciente
    df = df[~df.index.duplicated(keep="last")]
    if ELIMINAR_VELAS_SIN_VOLUMEN and "volume" in df.columns:
        df = df[df["volume"] > 0]
    descartadas = filas_iniciales - len(df)
    if descartadas:
        print(f"🧹 Se descartaron {descartadas:,} filas inválidas, duplicadas o sin volumen.")
    return df


try:
    ruta_csv = os.path.join(RUTA_ACTIVOS, NOMBRE_ARCHIVO)
    df_total = cargar_csv_dukascopy(ruta_csv)

    # Periodo del indicador más largo → número de velas a purgar al inicio del OOS
    PERIODO_PURGA = int(max(np.max(GRAN_MALLE_PARAMETROS["ma_lenta"]),
                            np.max(GRAN_MALLE_PARAMETROS["ma_rapida"]),
                            np.max(GRAN_MALLE_PARAMETROS["rsi_periodo"]) + 1))

    n_total = len(df_total)
    n_is = int(n_total * PORCENTAJE_IS)
    if n_is < PERIODO_PURGA * 3 or (n_total - n_is) < PERIODO_PURGA * 2:
        raise ValueError(
            f"❌ Muy pocos datos ({n_total:,} velas) para indicadores de hasta {PERIODO_PURGA} periodos. "
            "Usa un histórico más largo o reduce los periodos máximos de la malla."
        )

    # --- División estricta: el IS nunca ve datos del OOS -------------------------------
    df_is = df_total.iloc[:n_is].copy()
    # El OOS completo empieza justo después del IS. Sus indicadores se calculan SOLO con
    # datos OOS (nunca con el final del IS), así que sus primeras PERIODO_PURGA velas son
    # el "calentamiento" de los indicadores.
    df_oos_completo = df_total.iloc[n_is:].copy()
    # --- PURGA: eliminamos esas velas iniciales afectadas por el indicador más largo -----
    #     No se opera ni se mide nada en ellas → cero contaminación entre IS y OOS.
    df_oos = df_oos_completo.iloc[PERIODO_PURGA:].copy()

    # Frecuencia de las velas y nº de velas por año (para anualizar el Sharpe)
    FRECUENCIA = pd.Series(df_total.index).diff().median()
    anios = (df_total.index[-1] - df_total.index[0]).total_seconds() / (365.25 * 24 * 3600)
    BARRAS_POR_ANIO = float(n_total / anios) if anios > 0 else 252.0

    del df_total
    gc.collect()

    resumen = pd.DataFrame({
        "Velas": [len(df_is), PERIODO_PURGA, len(df_oos)],
        "Desde": [df_is.index[0], df_oos_completo.index[0], df_oos.index[0]],
        "Hasta": [df_is.index[-1], df_oos_completo.index[PERIODO_PURGA - 1], df_oos.index[-1]],
    }, index=["In-Sample (entrenamiento)", "Purga (calentamiento OOS)", "Out-Of-Sample (validación)"])
    print(f"✅ '{NOMBRE_ARCHIVO}' cargado: {n_total:,} velas | frecuencia {FRECUENCIA} | "
          f"~{BARRAS_POR_ANIO:,.0f} velas/año | RAM usada: "
          f"{(df_is.memory_usage().sum() + df_oos_completo.memory_usage().sum()) / 1024**2:,.1f} MB")
    display(resumen)
except (FileNotFoundError, ValueError) as error:
    raise SystemExit(str(error))
except Exception as error:
    raise SystemExit(f"❌ Error inesperado leyendo el CSV: {error}")

## 4️⃣ Motor de generación masiva acelerado por hardware (GPU)

**Estrategia:** cruce de medias móviles simples (rápida / lenta) con filtro RSI.

| Dirección | Entrada | Salida |
|---|---|---|
| `COMPRAS_SOLO` | MA rápida cruza **arriba** de la lenta y RSI < umbral | MA rápida cruza abajo |
| `VENTAS_SOLO` | MA rápida cruza **abajo** de la lenta y RSI > 100 − umbral | MA rápida cruza arriba |

**Cómo se usa la GPU:**
- **CuPy** calcula todas las SMA y RSI de la malla directamente en la VRAM.
- **Numba CUDA** compila un *kernel* donde **cada hilo de la GPU simula una estrategia completa**. Las señales se generan dentro del kernel, sin crear matrices gigantes en memoria.
- **Filtro de dirección estricto:** en `COMPRAS_SOLO` las entradas cortas (`short_entries`) son `False` absoluto; en `VENTAS_SOLO` lo son las entradas largas (`entries`).
- Las órdenes se ejecutan en la **apertura de la vela siguiente** a la señal, para no mirar el futuro.

**¿Por qué no `Portfolio.from_signals()` para toda la malla?** La versión gratuita de VectorBT simula en **CPU** (Numba) y necesitaría matrices de señales de decenas de GB para 100.000+ estrategias. Por eso la malla completa se simula en el kernel CUDA y VectorBT `from_signals()` se usa en la Celda 5 para las mejores estrategias. Al final de esta celda se comprueba que ambos motores dan **los mismos resultados**.

In [ ]:
# ==============================================================================
# CELDA 4 · MOTOR DE GENERACIÓN MASIVA ACELERADO POR GPU (CuPy + Numba CUDA)
# ==============================================================================
# Arquitectura (por qué está hecho así):
#   • VectorBT (versión open-source) simula carteras con Numba en CPU; no tiene backend
#     GPU y `vbt.Param` solo existe en VectorBT PRO. Meter 100.000+ columnas de señales en
#     `Portfolio.from_signals()` exigiría decenas de GB de matrices booleanas en RAM.
#   • Solución HPC: un KERNEL CUDA en el que cada hilo de la GPU simula UNA combinación de
#     parámetros sobre todas las velas (la RTX Pro 6000 ejecuta decenas de miles de hilos a
#     la vez). Las señales se calculan dentro del kernel → no se crean matrices gigantes.
#   • Los indicadores (SMA y RSI) se calculan una sola vez por periodo con CuPy en la GPU
#     mediante sumas acumuladas (100% vectorizado, sin bucles de Python).
#   • Después, VectorBT `Portfolio.from_signals()` vuelve a simular las mejores
#     estrategias (Celda 5) y aquí verificamos que el motor GPU coincide con VectorBT.
#   • Si no hay GPU, el mismo código se compila con Numba en CPU paralelo.
# ==============================================================================
import gc
import math
import time

import numpy as np
import pandas as pd
import vectorbt as vbt
from numba import njit, prange

try:
    import cupy as cp
except ImportError:
    cp = None

xp = cp if GPU_DISPONIBLE else np          # xp = "NumPy de la GPU" o NumPy normal
MOTOR = "GPU (Numba CUDA)" if NUMBA_CUDA_OK else "CPU paralelo (Numba)"
ES_COMPRA = DIRECCION_OPERACION == "COMPRAS_SOLO"
COLUMNAS_METRICAS = ["retorno_total", "profit_factor", "trades", "max_drawdown", "win_rate", "sharpe"]

if GPU_DISPONIBLE:
    # Límite de VRAM para CuPy: nunca más de FRACCION_MAX_VRAM de la memoria de la GPU
    cp.get_default_memory_pool().set_limit(fraction=FRACCION_MAX_VRAM)


def a_numpy(arr):
    """Copia un arreglo de la GPU a la RAM (si ya está en RAM no hace nada)."""
    return cp.asnumpy(arr) if (cp is not None and isinstance(arr, cp.ndarray)) else np.asarray(arr)


# ------------------------------------------------------------------------------
# 1) Indicadores vectorizados para TODOS los periodos a la vez (en la GPU)
# ------------------------------------------------------------------------------
def calcular_smas(close, periodos):
    """Matriz (n_periodos, n_velas) de medias móviles simples. NaN durante el calentamiento.
    Cada fila se calcula con una suma acumulada en la GPU: O(n) por periodo, sin bucles por vela."""
    c = xp.asarray(close, dtype=xp.float64)
    acumulada = xp.concatenate([xp.zeros(1, dtype=xp.float64), xp.cumsum(c)])
    salida = xp.full((len(periodos), c.shape[0]), xp.nan, dtype=xp.float32)
    for i, p in enumerate(periodos):
        p = int(p)
        salida[i, p - 1:] = ((acumulada[p:] - acumulada[:-p]) / p).astype(xp.float32)
    return salida


def calcular_rsis(close, periodos):
    """RSI con medias simples de ganancias/pérdidas (igual que vbt.RSI por defecto)."""
    c = xp.asarray(close, dtype=xp.float64)
    delta = xp.diff(c)
    acum_sube = xp.concatenate([xp.zeros(1), xp.cumsum(xp.maximum(delta, 0.0))])
    acum_baja = xp.concatenate([xp.zeros(1), xp.cumsum(xp.maximum(-delta, 0.0))])
    salida = xp.full((len(periodos), c.shape[0]), xp.nan, dtype=xp.float32)
    for i, p in enumerate(periodos):
        p = int(p)
        sube = acum_sube[p:] - acum_sube[:-p]       # suma de subidas de las últimas p velas
        baja = acum_baja[p:] - acum_baja[:-p]
        total = sube + baja
        rsi = 100.0 * sube / xp.where(total > 0, total, xp.nan)
        salida[i, p:] = rsi.astype(xp.float32)
    return salida


def construir_indicadores(df):
    """Calcula en la GPU todas las SMA y RSI necesarias para la malla, sobre `df`."""
    close = xp.asarray(df["close"].to_numpy(np.float32))
    return {
        "open": xp.asarray(df["open"].to_numpy(np.float32)),
        "close": close,
        "ma": calcular_smas(close, PERIODOS_MA),
        "rsi": calcular_rsis(close, PERIODOS_RSI),
    }


# ------------------------------------------------------------------------------
# 2) Malla de parámetros (equivalente a vbt.Param de VectorBT PRO, con producto cartesiano)
# ------------------------------------------------------------------------------
PERIODOS_MA = np.unique(np.concatenate([np.asarray(GRAN_MALLE_PARAMETROS["ma_rapida"]),
                                        np.asarray(GRAN_MALLE_PARAMETROS["ma_lenta"])])).astype(np.int64)
PERIODOS_RSI = np.unique(np.asarray(GRAN_MALLE_PARAMETROS["rsi_periodo"])).astype(np.int64)


def construir_malla():
    rap = np.asarray(GRAN_MALLE_PARAMETROS["ma_rapida"])
    len_ = np.asarray(GRAN_MALLE_PARAMETROS["ma_lenta"])
    rsi_p = np.asarray(GRAN_MALLE_PARAMETROS["rsi_periodo"])
    umb = np.asarray(GRAN_MALLE_PARAMETROS["rsi_umbral"], dtype=np.float32)
    # indexing="ij": el umbral varía más rápido → hilos vecinos de la GPU leen las mismas
    # medias móviles (accesos a memoria agrupados = más velocidad).
    R, L, P, U = np.meshgrid(rap, len_, rsi_p, umb, indexing="ij")
    R, L, P, U = R.ravel(), L.ravel(), P.ravel(), U.ravel()
    validas = R < L
    malla = pd.DataFrame({
        "ma_rapida": R[validas].astype(np.int32),
        "ma_lenta": L[validas].astype(np.int32),
        "rsi_periodo": P[validas].astype(np.int32),
        "rsi_umbral": U[validas].astype(np.float32),
    })
    malla["idx_rapida"] = np.searchsorted(PERIODOS_MA, malla["ma_rapida"]).astype(np.int32)
    malla["idx_lenta"] = np.searchsorted(PERIODOS_MA, malla["ma_lenta"]).astype(np.int32)
    malla["idx_rsi"] = np.searchsorted(PERIODOS_RSI, malla["rsi_periodo"]).astype(np.int32)
    return malla


# ------------------------------------------------------------------------------
# 3) Simulación de UNA combinación (se compila para GPU y para CPU con el mismo código)
# ------------------------------------------------------------------------------
#   Reglas de ejecución (idénticas a la validación con VectorBT):
#   • La señal se calcula con el CIERRE de la vela t-1 y se ejecuta en la APERTURA de la
#     vela t (sin mirar el futuro). La cartera se valora al cierre de cada vela.
#   • Se invierte todo el capital (size=inf) pagando COMISION_PCT en la entrada y la salida.
def _simular_combinacion(k, open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                         es_compra, comision, inicio, factor_anual, out):
    f = c_rap[k]
    s = c_len[k]
    r = c_rsi[k]
    umbral_compra = c_umb[k]
    umbral_venta = 100.0 - umbral_compra

    capital = 1.0            # capital realizado (normalizado a 1)
    valor_prev = 1.0         # valor de la cartera al cierre anterior
    pico = 1.0
    max_dd = 0.0
    en_posicion = False
    unidades = 0.0
    precio_entrada = 0.0
    capital_entrada = 0.0
    n_trades = 0
    n_ganadores = 0
    ganancias = 0.0
    perdidas = 0.0
    suma_r = 0.0
    suma_r2 = 0.0
    n_r = 0

    for t in range(inicio, close.shape[0]):
        j = t - 1                                    # vela de la señal
        mf = ma[f, j]
        ms = ma[s, j]
        mf_prev = ma[f, j - 1]
        ms_prev = ma[s, j - 1]
        cruce_arriba = (mf > ms) and (mf_prev <= ms_prev)   # NaN → False automáticamente
        cruce_abajo = (mf < ms) and (mf_prev >= ms_prev)
        valor_rsi = rsi[r, j]
        if es_compra:
            senal_entrada = cruce_arriba and (valor_rsi < umbral_compra)
            senal_salida = cruce_abajo
        else:
            senal_entrada = cruce_abajo and (valor_rsi > umbral_venta)
            senal_salida = cruce_arriba

        precio = open_[t]
        if en_posicion and senal_salida:
            if es_compra:
                capital = unidades * precio * (1.0 - comision)
            else:
                capital = capital_entrada + unidades * (precio_entrada * (1.0 - comision)
                                                        - precio * (1.0 + comision))
            pnl = capital - capital_entrada
            n_trades += 1
            if pnl > 0:
                n_ganadores += 1
                ganancias += pnl
            else:
                perdidas -= pnl
            en_posicion = False
        elif (not en_posicion) and senal_entrada and capital > 0:
            capital_entrada = capital
            precio_entrada = precio
            unidades = capital / (precio * (1.0 + comision))
            en_posicion = True

        # Valoración al cierre de la vela (mark-to-market)
        if en_posicion:
            if es_compra:
                valor = unidades * close[t]
            else:
                valor = capital_entrada + unidades * (precio_entrada * (1.0 - comision) - close[t])
        else:
            valor = capital

        if valor_prev > 0:
            ret = valor / valor_prev - 1.0
            suma_r += ret
            suma_r2 += ret * ret
            n_r += 1
        valor_prev = valor
        if valor > pico:
            pico = valor
        if pico > 0:
            dd = (pico - valor) / pico
            if dd > max_dd:
                max_dd = dd

    out[k, 0] = valor_prev - 1.0
    if perdidas > 0:
        out[k, 1] = ganancias / perdidas
    elif ganancias > 0:
        out[k, 1] = math.inf
    else:
        out[k, 1] = 0.0
    out[k, 2] = n_trades
    out[k, 3] = max_dd
    out[k, 4] = n_ganadores / n_trades if n_trades > 0 else math.nan
    if n_r > 1:
        media = suma_r / n_r
        var = (suma_r2 - suma_r * suma_r / n_r) / (n_r - 1)
        out[k, 5] = media / math.sqrt(var) * factor_anual if var > 1e-30 else math.nan
    else:
        out[k, 5] = math.nan


# --- Versión CPU (Numba paralelo: un núcleo por bloque de combinaciones) --------------
_simular_cpu = njit(cache=False)(_simular_combinacion)


@njit(parallel=True, cache=False)
def _backtest_cpu(open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                  es_compra, comision, inicio, factor_anual, out):
    for k in prange(out.shape[0]):
        _simular_cpu(k, open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                     es_compra, comision, inicio, factor_anual, out)


# --- Versión GPU (un hilo CUDA = una combinación) -------------------------------------
if NUMBA_CUDA_OK:
    from numba import cuda

    _simular_gpu = cuda.jit(device=True)(_simular_combinacion)

    @cuda.jit
    def _backtest_gpu(open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                      es_compra, comision, inicio, factor_anual, out):
        k = cuda.grid(1)
        if k < out.shape[0]:
            _simular_gpu(k, open_, close, ma, rsi, c_rap, c_len, c_rsi, c_umb,
                         es_compra, comision, inicio, factor_anual, out)


def backtest_masivo(ind, malla, inicio):
    """Simula todas las filas de `malla` y devuelve un DataFrame de métricas."""
    n_comb = len(malla)
    inicio = max(int(inicio), 2)
    factor_anual = math.sqrt(BARRAS_POR_ANIO)
    columnas = [malla[c].to_numpy() for c in ("idx_rapida", "idx_lenta", "idx_rsi", "rsi_umbral")]

    if NUMBA_CUDA_OK:
        # Tamaño de lote según la VRAM libre (16 B de parámetros + 48 B de métricas por combinación)
        libre, _ = cp.cuda.runtime.memGetInfo()
        lote = int(min(LOTE_MAX_COMBINACIONES, max(1, libre * FRACCION_MAX_VRAM // 64)))
        resultados = np.empty((n_comb, len(COLUMNAS_METRICAS)), dtype=np.float64)
        hilos = 128
        for a in range(0, n_comb, lote):
            b = min(a + lote, n_comb)
            params_gpu = [cp.asarray(c[a:b]) for c in columnas]
            out_gpu = cp.empty((b - a, len(COLUMNAS_METRICAS)), dtype=cp.float64)
            bloques = (b - a + hilos - 1) // hilos
            _backtest_gpu[bloques, hilos](ind["open"], ind["close"], ind["ma"], ind["rsi"],
                                          *params_gpu, ES_COMPRA, COMISION_PCT, inicio,
                                          factor_anual, out_gpu)
            cuda.synchronize()
            resultados[a:b] = cp.asnumpy(out_gpu)
            del params_gpu, out_gpu
            print(f"   ⚡ Lote {a:,}–{b:,} de {n_comb:,} completado")
        cp.get_default_memory_pool().free_all_blocks()
    else:
        resultados = np.empty((n_comb, len(COLUMNAS_METRICAS)), dtype=np.float64)
        _backtest_cpu(a_numpy(ind["open"]), a_numpy(ind["close"]), a_numpy(ind["ma"]),
                      a_numpy(ind["rsi"]), *columnas, ES_COMPRA, COMISION_PCT, inicio,
                      factor_anual, resultados)

    met = pd.DataFrame(resultados, columns=COLUMNAS_METRICAS, index=malla.index)
    met["trades"] = met["trades"].astype(np.int64)
    met["calmar"] = met["retorno_total"] / met["max_drawdown"].replace(0, np.nan)
    return met


# ------------------------------------------------------------------------------
# 4) Simulación oficial con VectorBT Portfolio.from_signals (para Top y validación)
# ------------------------------------------------------------------------------
def _desplazar_1(m):
    """Retrasa las señales una vela: señal al cierre de t-1 → orden en la apertura de t."""
    out = np.zeros_like(m)
    out[:, 1:] = m[:, :-1]
    return out


def generar_senales(ind_np, combos):
    """Matrices de señales (velas × combinaciones) totalmente vectorizadas con NumPy."""
    ma, rsi = ind_np["ma"], ind_np["rsi"]
    f = ma[combos["idx_rapida"].to_numpy()]
    s = ma[combos["idx_lenta"].to_numpy()]
    r = rsi[combos["idx_rsi"].to_numpy()]
    u = combos["rsi_umbral"].to_numpy(np.float32)[:, None]
    f_prev = np.full_like(f, np.nan)
    f_prev[:, 1:] = f[:, :-1]
    s_prev = np.full_like(s, np.nan)
    s_prev[:, 1:] = s[:, :-1]
    with np.errstate(invalid="ignore"):
        cruce_arriba = (f > s) & (f_prev <= s_prev)
        cruce_abajo = (f < s) & (f_prev >= s_prev)
        if ES_COMPRA:
            entradas, salidas = cruce_arriba & (r < u), cruce_abajo
        else:
            entradas, salidas = cruce_abajo & (r > 100.0 - u), cruce_arriba
    return _desplazar_1(entradas).T, _desplazar_1(salidas).T


def simular_vectorbt(df, ind_np, combos, inicio=0):
    """Simula `combos` con VectorBT desde la vela `inicio` y devuelve (métricas, portfolio)."""
    inicio = max(int(inicio), 2)
    entradas, salidas = generar_senales(ind_np, combos)
    entradas, salidas = entradas[inicio:], salidas[inicio:]
    vacias = np.zeros_like(entradas)            # Matriz de False absolutos
    indice = df.index[inicio:]
    columnas = pd.RangeIndex(len(combos))
    a_df = lambda m: pd.DataFrame(m, index=indice, columns=columnas)

    if ES_COMPRA:   # COMPRAS_SOLO → short_entries forzadas a False absoluto
        largas_e, largas_s, cortas_e, cortas_s = entradas, salidas, vacias, vacias
    else:           # VENTAS_SOLO → entries (largas) forzadas a False absoluto
        largas_e, largas_s, cortas_e, cortas_s = vacias, vacias, entradas, salidas

    pf = vbt.Portfolio.from_signals(
        close=df["close"].iloc[inicio:],
        price=df["open"].iloc[inicio:],          # Ejecución en la apertura de la vela
        entries=a_df(largas_e), exits=a_df(largas_s),
        short_entries=a_df(cortas_e), short_exits=a_df(cortas_s),
        size=np.inf, fees=COMISION_PCT, init_cash=CAPITAL_INICIAL, freq=FRECUENCIA,
    )
    rets = pf.returns()
    cerrados = pf.trades.closed
    met = pd.DataFrame({
        "retorno_total": pf.total_return().to_numpy(),
        "profit_factor": cerrados.profit_factor().to_numpy(),
        "trades": cerrados.count().to_numpy(),
        "max_drawdown": np.abs(pf.max_drawdown().to_numpy()),
        "win_rate": cerrados.win_rate().to_numpy(),
        "sharpe": (rets.mean() / rets.std(ddof=1) * np.sqrt(BARRAS_POR_ANIO)).to_numpy(),
    }, index=combos.index)
    met["calmar"] = met["retorno_total"] / met["max_drawdown"].replace(0, np.nan)
    return met, pf


# ==============================================================================
# EJECUCIÓN: malla completa sobre el bloque In-Sample
# ==============================================================================
try:
    print(f"🚀 Motor de simulación: {MOTOR} | Indicadores: {'CuPy (GPU)' if GPU_DISPONIBLE else 'NumPy (CPU)'}")
    t0 = time.time()
    malla = construir_malla()
    ind_is = construir_indicadores(df_is)          # Indicadores SOLO con datos In-Sample
    if GPU_DISPONIBLE:
        cp.cuda.Stream.null.synchronize()
    print(f"   📐 {len(PERIODOS_MA)} SMAs + {len(PERIODOS_RSI)} RSIs calculadas en {time.time() - t0:.2f} s "
          f"({ind_is['ma'].nbytes / 1024**2:,.1f} MB en {'VRAM' if GPU_DISPONIBLE else 'RAM'})")

    t1 = time.time()
    metricas_is = backtest_masivo(ind_is, malla, inicio=2)
    dt = time.time() - t1
    resultados_is = pd.concat([malla[["ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"]], metricas_is], axis=1)
    velas_simuladas = len(malla) * len(df_is)
    print(f"✅ {len(malla):,} estrategias simuladas sobre {len(df_is):,} velas en {dt:,.2f} s "
          f"→ {velas_simuladas / max(dt, 1e-9) / 1e9:,.2f} mil millones de velas·estrategia/seg")

    # --- Verificación de coherencia: motor GPU vs VectorBT en 5 combinaciones al azar ----
    activas = resultados_is[resultados_is["trades"] > 0]
    muestra = (activas if len(activas) else resultados_is).sample(
        min(5, len(activas) or len(resultados_is)), random_state=42).index
    ind_is_np = {k: a_numpy(v) for k, v in ind_is.items()}
    met_vbt, _ = simular_vectorbt(df_is, ind_is_np, malla.loc[muestra], inicio=2)
    diferencia = (met_vbt["retorno_total"] - metricas_is.loc[muestra, "retorno_total"]).abs().max()
    trades_ok = (met_vbt["trades"].to_numpy() == metricas_is.loc[muestra, "trades"].to_numpy()).all()
    estado = "✅ coincide" if (diferencia < 1e-3 and trades_ok) else "⚠️ revisar"
    print(f"🔎 Verificación {MOTOR} vs VectorBT: {estado} "
          f"(máx. diferencia de retorno = {diferencia:.2e}, nº trades idéntico = {trades_ok})")

    del ind_is
    gc.collect()
    if GPU_DISPONIBLE:
        cp.get_default_memory_pool().free_all_blocks()
except Exception as error:
    raise SystemExit(f"❌ Error en el motor de simulación: {error}\n"
                     "   Si es un error de memoria, reduce GRAN_MALLE_PARAMETROS o LOTE_MAX_COMBINACIONES.")

## 5️⃣ Filtrado automático de descarte y validación Out-Of-Sample

1. **Descarte masivo:** se eliminan automáticamente todas las estrategias con retorno ≤ 0, Profit Factor bajo, pocas o demasiadas operaciones, o un drawdown excesivo.
2. **Top In-Sample:** las supervivientes se ordenan por `METRICA_RANKING` y se quedan las `TOP_N` mejores.
3. **Validación OOS:** ese Top se vuelve a simular con **VectorBT `Portfolio.from_signals()`** en el bloque oculto Out-Of-Sample.
4. La tabla final muestra **solo** las estrategias que siguen ganando en OOS. `robustez_sharpe` ≈ 1 significa que la estrategia rinde fuera de muestra igual que dentro (muy buena señal); si se acerca a 0, la estrategia estaba sobreajustada.
5. **Mapa de calor:** zonas verdes amplias indican parámetros robustos; un punto verde aislado suele ser sobreajuste.

In [ ]:
# ==============================================================================
# CELDA 5 · FILTRO DE DESCARTE MASIVO + TOP IN-SAMPLE + VALIDACIÓN OUT-OF-SAMPLE
# ==============================================================================
import matplotlib.pyplot as plt

try:
    # ------------------------------------------------------------------------------
    # 1) Descarte automático (máscaras booleanas vectorizadas sobre TODAS las combinaciones)
    # ------------------------------------------------------------------------------
    fd = FILTROS_DESCARTE
    reglas = {
        f"Retorno total <= {fd['retorno_min']:.0%}":      resultados_is["retorno_total"] <= fd["retorno_min"],
        f"Profit Factor < {fd['profit_factor_min']}":     resultados_is["profit_factor"] < fd["profit_factor_min"],
        f"Trades < {fd['trades_min']}":                   resultados_is["trades"] < fd["trades_min"],
        f"Trades > {fd['trades_max']}":                   resultados_is["trades"] > fd["trades_max"],
        f"Drawdown > {fd['drawdown_max']:.0%}":           resultados_is["max_drawdown"] > fd["drawdown_max"],
        "Sharpe no calculable":                           resultados_is["sharpe"].isna(),
    }
    descartar = np.zeros(len(resultados_is), dtype=bool)
    print(f"🧪 Filtro de descarte sobre {len(resultados_is):,} estrategias In-Sample:")
    for nombre, mascara in reglas.items():
        nuevas = int((mascara.to_numpy() & ~descartar).sum())
        descartar |= mascara.to_numpy()
        print(f"   ✂️  {nombre:<28} → {nuevas:>10,} eliminadas")
    supervivientes = resultados_is.loc[~descartar]
    print(f"   🏆 Supervivientes: {len(supervivientes):,} "
          f"({len(supervivientes) / max(len(resultados_is), 1):.2%} del total)")

    if supervivientes.empty:
        raise SystemExit(
            "❌ Ninguna estrategia superó los filtros In-Sample. Prueba a relajar FILTROS_DESCARTE, "
            "ampliar GRAN_MALLE_PARAMETROS o cambiar DIRECCION_OPERACION."
        )

    # ------------------------------------------------------------------------------
    # 2) Top N In-Sample y simulación oficial con VectorBT (IS y OOS)
    # ------------------------------------------------------------------------------
    top_is = supervivientes.sort_values(METRICA_RANKING, ascending=False).head(TOP_N)
    combos_top = malla.loc[top_is.index]

    # In-Sample con VectorBT (métricas oficiales)
    ind_is_np = {k: a_numpy(v) for k, v in construir_indicadores(df_is).items()}
    met_is_vbt, pf_is = simular_vectorbt(df_is, ind_is_np, combos_top, inicio=2)

    # Out-Of-Sample: indicadores calculados SOLO con datos OOS; las primeras PERIODO_PURGA
    # velas (purga/calentamiento) no se operan ni se miden.
    ind_oos_np = {k: a_numpy(v) for k, v in construir_indicadores(df_oos_completo).items()}
    met_oos_vbt, pf_oos = simular_vectorbt(df_oos_completo, ind_oos_np, combos_top, inicio=PERIODO_PURGA)

    # ------------------------------------------------------------------------------
    # 3) Tabla comparativa IS vs OOS (solo estrategias que superan la validación)
    # ------------------------------------------------------------------------------
    fo = FILTROS_OOS
    comparativa = pd.concat([
        combos_top[["ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral"]],
        met_is_vbt.add_suffix("_IS"),
        met_oos_vbt.add_suffix("_OOS"),
    ], axis=1)
    comparativa["supera_OOS"] = (
        (comparativa["retorno_total_OOS"] > fo["retorno_min"])
        & (comparativa["profit_factor_OOS"] >= fo["profit_factor_min"])
        & (comparativa["trades_OOS"] >= fo["trades_min"])
    )
    # Ratio de robustez: qué fracción del Sharpe In-Sample se conserva fuera de muestra
    comparativa["robustez_sharpe"] = comparativa["sharpe_OOS"] / comparativa["sharpe_IS"]
    comparativa.insert(0, "ranking_IS", np.arange(1, len(comparativa) + 1))
    estrategias_validas = comparativa[comparativa["supera_OOS"]].copy()

    print(f"\n🔬 Validación Out-Of-Sample del Top {len(top_is)} (VectorBT Portfolio.from_signals):")
    print(f"   ✅ Robustas (superan OOS): {len(estrategias_validas)}")
    print(f"   ❌ Sobreoptimizadas (fallan en OOS): {len(comparativa) - len(estrategias_validas)}")

    if estrategias_validas.empty:
        print("\n⚠️ Ninguna estrategia del Top superó el Out-Of-Sample: señal clara de sobreajuste.\n"
              "   Prueba con más datos, otra dirección operativa o filtros IS más exigentes.")
    else:
        columnas_tabla = ["ranking_IS", "ma_rapida", "ma_lenta", "rsi_periodo", "rsi_umbral",
                          "retorno_total_IS", "retorno_total_OOS", "profit_factor_IS", "profit_factor_OOS",
                          "sharpe_IS", "sharpe_OOS", "max_drawdown_IS", "max_drawdown_OOS",
                          "trades_IS", "trades_OOS", "win_rate_IS", "win_rate_OOS", "robustez_sharpe"]
        formato = {c: "{:.2%}" for c in columnas_tabla if c.startswith(("retorno", "max_drawdown", "win_rate"))}
        formato.update({c: "{:.2f}" for c in columnas_tabla if c.startswith(("profit", "sharpe", "robustez"))})
        formato["rsi_umbral"] = "{:.0f}"
        try:
            display(estrategias_validas[columnas_tabla].style.format(formato)
                    .background_gradient(subset=["sharpe_OOS"], cmap="RdYlGn").hide(axis="index"))
        except Exception:   # Sin jinja2 → tabla simple
            display(estrategias_validas[columnas_tabla].round(4))

    # ------------------------------------------------------------------------------
    # 4) Gráficos: mapa de calor de parámetros y curva de capital de la mejor estrategia
    # ------------------------------------------------------------------------------
    if MOSTRAR_GRAFICOS:
        metrica_mapa = METRICA_RANKING
        mapa = resultados_is.pivot_table(index="ma_lenta", columns="ma_rapida",
                                         values=metrica_mapa, aggfunc="max")
        mapa = mapa.replace([np.inf, -np.inf], np.nan)
        fig, ax = plt.subplots(figsize=(14, 7))
        lim = np.nanpercentile(np.abs(mapa.to_numpy()), 98) if np.isfinite(mapa.to_numpy()).any() else 1
        im = ax.imshow(mapa.to_numpy(), aspect="auto", origin="lower", cmap="RdYlGn", vmin=-lim, vmax=lim,
                       extent=[mapa.columns.min(), mapa.columns.max(), mapa.index.min(), mapa.index.max()])
        fig.colorbar(im, ax=ax, label=f"{metrica_mapa} (mejor valor por par de medias)")
        ax.set_xlabel("Periodo MA rápida")
        ax.set_ylabel("Periodo MA lenta")
        ax.set_title(f"Mapa de calor In-Sample · {NOMBRE_ARCHIVO} · {DIRECCION_OPERACION}\n"
                     "Zonas verdes amplias = parámetros robustos; puntos aislados = posible sobreajuste")
        plt.show()

        if not estrategias_validas.empty:
            mejor = estrategias_validas.index[0]
            pos = combos_top.index.get_loc(mejor)
            curva_is = pf_is.value().iloc[:, pos]
            curva_oos = pf_oos.value().iloc[:, pos] / pf_oos.value().iloc[0, pos] * curva_is.iloc[-1]
            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(curva_is.index, curva_is.to_numpy(), color="#2563eb", label="In-Sample")
            ax.plot(curva_oos.index, curva_oos.to_numpy(), color="#16a34a", label="Out-Of-Sample")
            ax.axvline(df_oos.index[0], color="gray", linestyle="--", linewidth=1)
            fila = estrategias_validas.loc[mejor]
            ax.set_title(f"Curva de capital · MA {fila.ma_rapida:.0f}/{fila.ma_lenta:.0f} · "
                         f"RSI {fila.rsi_periodo:.0f} < {fila.rsi_umbral:.0f}")
            ax.set_ylabel("Capital")
            ax.legend()
            ax.grid(alpha=0.3)
            plt.show()

    del ind_is_np, ind_oos_np
    gc.collect()
except SystemExit:
    raise
except Exception as error:
    raise SystemExit(f"❌ Error en el filtrado o la validación OOS: {error}")

## 6️⃣ Exportación de resultados a Google Drive

Guarda **solo** las estrategias que superaron el filtro In-Sample **y** la validación Out-Of-Sample en:

`/content/drive/MyDrive/activos/resultados_[NOMBRE]_[DIRECCION].csv`

In [ ]:
# ==============================================================================
# CELDA 6 · EXPORTACIÓN DE RESULTADOS A GOOGLE DRIVE
# ==============================================================================
try:
    if "estrategias_validas" not in globals():
        raise RuntimeError("Primero ejecuta correctamente las Celdas 1 a 5.")
    if estrategias_validas.empty:
        print("ℹ️ No hay estrategias que hayan superado los filtros IS y la validación OOS: "
              "no se genera archivo.")
    else:
        nombre_base = os.path.splitext(NOMBRE_ARCHIVO)[0]
        ruta_salida = os.path.join(RUTA_ACTIVOS, f"resultados_{nombre_base}_{DIRECCION_OPERACION}.csv")
        exportar = estrategias_validas.drop(columns=["supera_OOS"]).copy()
        exportar.insert(0, "activo", nombre_base)
        exportar.insert(1, "direccion", DIRECCION_OPERACION)
        exportar["comision_pct"] = COMISION_PCT
        exportar["inicio_IS"] = df_is.index[0]
        exportar["fin_IS"] = df_is.index[-1]
        exportar["inicio_OOS"] = df_oos.index[0]
        exportar["fin_OOS"] = df_oos.index[-1]
        exportar = exportar.replace([np.inf, -np.inf], np.nan).round(6)
        exportar.to_csv(ruta_salida, index=False, encoding="utf-8")
        print(f"💾 {len(exportar)} estrategias robustas guardadas en:\n   {ruta_salida}")
        display(exportar.head(10))
except Exception as error:
    raise SystemExit(f"❌ No se pudo guardar el CSV en Google Drive: {error}\n"
                     "   Comprueba que Drive sigue montado (vuelve a ejecutar la Celda 1).")